# Reviewer 3 — Comment 7: protein FG20 audit (verified, with canonical-structure control)

**Source:** FG20 SMARTS and original amino-acid SMILES dictionary reproduced in the historical preprocessing-code reconstruction supplied with the manuscript. The historical SMILES contain connectivity errors (e.g., the entry labeled alanine is not alanine). This notebook independently checks the output against RDKit's correctly constructed free amino acids (`Chem.MolFromSequence`) with the same SMARTS and the same cleanup rules.

The original encoding uses whole, isolated free amino acids, including common amino/carboxyl groups, not isolated side chains. Treat this as a recovered-implementation audit: before asserting that these were the precise features consumed during historical model training, confirm that the original trained preprocessing matches the recovered code.

**Interpretation:** Zero columns are constant, not stochastic noise. `log2(number_of_distinct_vectors)` is a maximum categorical capacity; Shannon entropy is calculated separately assuming uniform amino-acid identity. FG20 is deterministic conditional on complete amino-acid one-hot identity and adds zero *independent* bits. Neither feature audit nor an apparent KIBA CI decline proves a causal connection without matched KIBA ablation.


In [1]:
from collections import defaultdict, Counter
from math import log2
import numpy as np
from rdkit import Chem, rdBase

FG=(('CarboxylicAcid','[CX3](=O)[OX2H1]'),('Ester','[CX3](=O)[OX2][#6]'),('Amide','[NX3][CX3](=O)[#6]'),('Anhydride','[CX3](=O)O[CX3](=O)'),('AcylHalide','[CX3](=O)[Cl,Br,I,F]'),('Aldehyde','[CX3H1](=O)[#6]'),('Ketone','[#6][CX3](=O)[#6]'),('Alcohol','[#6;!a][OX2H]'),('Phenol','c[OX2H]'),('Ether','[OX2]([#6])[#6]'),('Nitrile','[CX2]#N'),('Nitro','[$([NX3](=O)=O),$([NX3+](=O)[O-])]'),('Amine_Primary','[NX3;H2][#6]'),('Amine_Secondary','[NX3;H1]([#6])[#6]'),('Amine_Tertiary','[NX3]([#6])([#6])[#6]'),('Thiol','[#16X2H]'),('Thioether','[#16X2]([#6])[#6]'),('Sulfoxide','[#16X3](=O)([#6])[#6]'),('Sulfone','[#16X4](=O)(=O)([#6])[#6]'),('Aryl','c1ccccc1'))
N=tuple(x[0] for x in FG)
P=[(n,Chem.MolFromSmarts(s)) for n,s in FG]
AA={'A':'NCC(C)C(=O)O','R':'NCC(CCCNC(N)=N)C(=O)O','N':'NCC(C(=O)N)C(=O)O','D':'NCC(C(=O)O)C(=O)O','C':'NCC(S)C(=O)O','E':'NCC(CCC(=O)O)C(=O)O','Q':'NCC(CCC(=O)N)C(=O)O','G':'NCC(=O)O','H':'NCC(Cc1c[nH]cn1)C(=O)O','I':'NCC(C(C)CC)C(=O)O','L':'NCC(CC(C)C)C(=O)O','K':'NCC(CCCCN)C(=O)O','M':'NCC(CCSC)C(=O)O','F':'NCC(Cc1ccccc1)C(=O)O','P':'N1CCC(C(=O)O)C1','S':'NCC(CO)C(=O)O','T':'NCC(C(O)C)C(=O)O','W':'NCC(Cc1c2ccccc2[nH]c1)C(=O)O','Y':'NCC(Cc1ccc(O)cc1)C(=O)O','V':'NCC(C(C)C)C(=O)O'}

def groups(m):
    g={n for n,p in P if m.HasSubstructMatch(p)}
    if 'Ester' in g:g.discard('Ether')
    if 'CarboxylicAcid' in g:g.discard('Alcohol')
    if 'Phenol' in g:g.discard('Alcohol')
    return g

rows=[]
for aa,smi in AA.items():
    g=groups(Chem.MolFromSmiles(smi)); v=np.array([int(n in g) for n in N]); rows.append((aa,smi,v))
mat=np.stack([v for _,_,v in rows])
zero=[N[i] for i in range(20) if mat[:,i].sum()==0]
clusters=defaultdict(list)
for aa,_,v in rows: clusters[tuple(v.tolist())].append(aa)
counts=Counter(tuple(v.tolist()) for _,_,v in rows)
ent=-sum((c/20)*log2(c/20) for c in counts.values())
summary={'rdkit_version':rdBase.rdkitVersion,'zero_columns_count':len(zero),'zero_columns':zero,'distinct_vectors':len(clusters),'max_categorical_capacity_bits_log2_distinct':log2(len(clusters)),'empirical_shannon_entropy_uniform_residues_bits':ent,'collision_groups':[x for x in clusters.values() if len(x)>1]}
summary

{'rdkit_version': '2025.09.4',
 'zero_columns_count': 12,
 'zero_columns': ['Ester',
  'Anhydride',
  'AcylHalide',
  'Aldehyde',
  'Ketone',
  'Alcohol',
  'Ether',
  'Nitrile',
  'Nitro',
  'Amine_Tertiary',
  'Sulfoxide',
  'Sulfone'],
 'distinct_vectors': 8,
 'max_categorical_capacity_bits_log2_distinct': 3.0,
 'empirical_shannon_entropy_uniform_residues_bits': 2.219240704636849,
 'collision_groups': [['A', 'D', 'E', 'G', 'H', 'I', 'L', 'K', 'S', 'T', 'V'],
  ['N', 'Q'],
  ['F', 'W']]}

In [2]:
import json
print(json.dumps(summary, indent=2))

{
  "rdkit_version": "2025.09.4",
  "zero_columns_count": 12,
  "zero_columns": [
    "Ester",
    "Anhydride",
    "AcylHalide",
    "Aldehyde",
    "Ketone",
    "Alcohol",
    "Ether",
    "Nitrile",
    "Nitro",
    "Amine_Tertiary",
    "Sulfoxide",
    "Sulfone"
  ],
  "distinct_vectors": 8,
  "max_categorical_capacity_bits_log2_distinct": 3.0,
  "empirical_shannon_entropy_uniform_residues_bits": 2.219240704636849,
  "collision_groups": [
    [
      "A",
      "D",
      "E",
      "G",
      "H",
      "I",
      "L",
      "K",
      "S",
      "T",
      "V"
    ],
    [
      "N",
      "Q"
    ],
    [
      "F",
      "W"
    ]
  ]
}


In [3]:
for sig, aas in sorted(clusters.items(), key=lambda x:(-len(x[1]), x[1])):
    active=[N[i] for i,x in enumerate(sig) if x]
    print(''.join(aas), '->', active)

ADEGHILKSTV -> ['CarboxylicAcid', 'Amine_Primary']
FW -> ['CarboxylicAcid', 'Amine_Primary', 'Aryl']
NQ -> ['CarboxylicAcid', 'Amide', 'Amine_Primary']
C -> ['CarboxylicAcid', 'Amine_Primary', 'Thiol']
M -> ['CarboxylicAcid', 'Amine_Primary', 'Thioether']
P -> ['CarboxylicAcid', 'Amine_Secondary']
R -> ['CarboxylicAcid', 'Amine_Primary', 'Amine_Secondary']
Y -> ['CarboxylicAcid', 'Phenol', 'Amine_Primary', 'Aryl']


In [4]:
# Chemically correct amino-acid reference, contrasted with original encoding.
from rdkit.Chem import rdMolDescriptors
FG_SMARTS=FG
FG_NAMES=N
AA_SMILES_NOTEBOOK=AA
COMPILED=P

def corrected_groups(m):
    names={name for name,pattern in COMPILED if m.HasSubstructMatch(pattern)}
    if 'Ester' in names: names.discard('Ether')
    if 'CarboxylicAcid' in names: names.discard('Alcohol')
    if 'Phenol' in names: names.discard('Alcohol')
    return names

def encoding(m):
    names=corrected_groups(m)
    return tuple(int(name in names) for name in FG_NAMES)

def connectivity(m):
    copy=Chem.Mol(m)
    Chem.RemoveStereochemistry(copy)
    return Chem.MolToSmiles(copy)

historical={aa:Chem.MolFromSmiles(s) for aa,s in AA_SMILES_NOTEBOOK.items()}
canonical={aa:Chem.MolFromSequence(aa) for aa in AA_SMILES_NOTEBOOK}
assert len(historical)==len(canonical)==20
assert all(m is not None for m in [*historical.values(),*canonical.values()])
wrong=[aa for aa in historical if connectivity(historical[aa])!=connectivity(canonical[aa])]
wrong_formula=[aa for aa in historical if rdMolDescriptors.CalcMolFormula(historical[aa])!=rdMolDescriptors.CalcMolFormula(canonical[aa])]
M_old=np.stack([encoding(historical[aa]) for aa in historical])
M_new=np.stack([encoding(canonical[aa]) for aa in canonical])
print('RDKit:',rdBase.rdkitVersion)
print('Different connectivity from canonical amino acid:',len(wrong),'/20:',','.join(wrong))
print('Different molecular formula:',len(wrong_formula),'/20:',','.join(wrong_formula))
print('Alanine historical:',Chem.MolToSmiles(historical['A']),'canonical:',Chem.MolToSmiles(canonical['A']))
print('Identical historical and canonical FG20 matrices:',np.array_equal(M_old,M_new))
assert np.array_equal(M_old,M_new), 'Chemically corrected structures change the FG20 encodings; report matrices separately.'


RDKit: 2025.09.4
Different connectivity from canonical amino acid: 19 /20: A,R,N,D,C,E,Q,H,I,L,K,M,F,P,S,T,W,Y,V
Different molecular formula: 15 /20: A,R,E,Q,H,I,L,K,M,F,S,T,W,Y,V
Alanine historical: CC(CN)C(=O)O canonical: C[C@H](N)C(=O)O
Identical historical and canonical FG20 matrices: True


In [5]:
# Report measurable coverage, information content, and residue collisions.
from collections import defaultdict,Counter
from math import log2
X=M_new
zero=[FG_NAMES[j] for j in range(20) if not X[:,j].any()]
ones=[FG_NAMES[j] for j in range(20) if X[:,j].all()]
variable=[FG_NAMES[j] for j in range(20) if X[:,j].any() and not X[:,j].all()]
collisions=defaultdict(list)
for aa,row in zip(AA_SMILES_NOTEBOOK,X): collisions[tuple(row.tolist())].append(aa)
entropy=-sum((len(v)/20)*log2(len(v)/20) for v in collisions.values())
print('Constant zero:',len(zero),'/20 =',len(zero)*5,'%')
print('Zero columns:',', '.join(zero))
print('Constant one:',len(ones),'; variable:',len(variable))
print('Distinct vectors:',len(collisions),'/20; max categorical capacity:',log2(len(collisions)),'bits')
print('Shannon entropy under uniform amino acids:',round(entropy,6),'bits; log2(20):',round(log2(20),6))
print('New information given complete one-hot residue identity: 0 bits (deterministic mapping)')
print('Residues with all-zero vector:', ''.join(aa for aa,row in zip(AA_SMILES_NOTEBOOK,X) if not row.any()) or 'NONE')
for signature,aas in sorted(collisions.items(),key=lambda item:(-len(item[1]),''.join(item[1]))):
    print(''.join(aas),'->',', '.join(FG_NAMES[j] for j,flag in enumerate(signature) if flag))
assert len(zero)==12 and len(collisions)==8 and len(ones)==1 and np.array_equal(M_old,M_new)
print('PASS: matrix, coverage, discriminability and corrected-structure controls')


Constant zero: 12 /20 = 60 %
Zero columns: Ester, Anhydride, AcylHalide, Aldehyde, Ketone, Alcohol, Ether, Nitrile, Nitro, Amine_Tertiary, Sulfoxide, Sulfone
Constant one: 1 ; variable: 7
Distinct vectors: 8 /20; max categorical capacity: 3.0 bits
Shannon entropy under uniform amino acids: 2.219241 bits; log2(20): 4.321928
New information given complete one-hot residue identity: 0 bits (deterministic mapping)
Residues with all-zero vector: NONE
ADEGHILKSTV -> CarboxylicAcid, Amine_Primary
FW -> CarboxylicAcid, Amine_Primary, Aryl
NQ -> CarboxylicAcid, Amide, Amine_Primary
C -> CarboxylicAcid, Amine_Primary, Thiol
M -> CarboxylicAcid, Amine_Primary, Thioether
P -> CarboxylicAcid, Amine_Secondary
R -> CarboxylicAcid, Amine_Primary, Amine_Secondary
Y -> CarboxylicAcid, Phenol, Amine_Primary, Aryl
PASS: matrix, coverage, discriminability and corrected-structure controls


In [6]:
# Reproducible per-residue FG20 matrix, also exportable as a CSV.
import pandas as pd
from pathlib import Path
rows_csv=[]
for aa,mol,flags in zip(AA_SMILES_NOTEBOOK,canonical.values(),X):
    row={'residue':aa,'corrected_smiles':Chem.MolToSmiles(mol),'active_patterns':';'.join(FG_NAMES[j] for j,bit in enumerate(flags) if bit)}
    row.update({f'FG_{name}':int(flags[j]) for j,name in enumerate(FG_NAMES)})
    rows_csv.append(row)
df=pd.DataFrame(rows_csv)
assert df.shape==(20,23)
df.to_csv('protein_fg20_amino_acid_matrix.csv',index=False)
print('Saved protein_fg20_amino_acid_matrix.csv; rows:',len(df),'FG columns:',len(FG_NAMES))


Saved protein_fg20_amino_acid_matrix.csv; rows: 20 FG columns: 20


## Reviewer-specific cross-check and defensible interpretation

The following checks distinguish the **correct, general redundancy argument** from reviewer-specific numerical claims. The free-amino-acid representation gives every residue a backbone-derived carboxylic-acid bit, so the seven named residues are **not literally all-zero** under this implementation. A different side-chain-only convention might yield different counts. The DAVIS ablation values are *transcribed from Table 11*, not predictions generated by this notebook; the arithmetic supports a possible learning/inductive-bias effect while establishing neither a new-information gain nor causality on KIBA. No model was retrained.


In [7]:
# Reviewer-specific falsification check for the RECOVERED WHOLE-RESIDUE encoding.
# This does not claim that a side-chain-only encoding yields the same answer.
reviewer_all_zero_examples = ('A','G','H','I','L','P','V')
for aa in reviewer_all_zero_examples:
    bits = M_new[list(AA_SMILES_NOTEBOOK).index(aa)]
    active = [FG_NAMES[i] for i,bit in enumerate(bits) if bit]
    print(f'{aa}: active FG20 bits = {active}')
    assert any(bits), f'Unexpected all-zero vector for {aa}'
assert all(M_new[:, FG_NAMES.index('CarboxylicAcid')] == 1)
print('PASS: all seven examples are nonzero under free-amino-acid/cleanup encoding.')
print('NOTE: this does NOT disprove a separate side-chain-only analysis by the reviewer.')


A: active FG20 bits = ['CarboxylicAcid', 'Amine_Primary']
G: active FG20 bits = ['CarboxylicAcid', 'Amine_Primary']
H: active FG20 bits = ['CarboxylicAcid', 'Amine_Primary']
I: active FG20 bits = ['CarboxylicAcid', 'Amine_Primary']
L: active FG20 bits = ['CarboxylicAcid', 'Amine_Primary']
P: active FG20 bits = ['CarboxylicAcid', 'Amine_Secondary']
V: active FG20 bits = ['CarboxylicAcid', 'Amine_Primary']
PASS: all seven examples are nonzero under free-amino-acid/cleanup encoding.
NOTE: this does NOT disprove a separate side-chain-only analysis by the reviewer.


In [8]:
# Arithmetic cross-check of the DAVIS Table 11 numbers; NOT a model rerun.
# Manually transcribed published ablation results (same stated split/settings).
import math
mse_graph_only=0.2018
mse_protein_fg_only=0.1822
mse_ligand_fg_only=0.1648
mse_both_fg=0.1546
print('DAVIS Table 11: graph only',mse_graph_only,'; protein FG only',mse_protein_fg_only,
      '; ligand FG only',mse_ligand_fg_only,'; both FG',mse_both_fg)
print('Adding protein FG to ligand FG: MSE change',round(mse_both_fg-mse_ligand_fg_only,4),
      '; percent change',round(100*(mse_both_fg/mse_ligand_fg_only-1),3),'%')
print('Adding protein FG to graph only: MSE change',round(mse_protein_fg_only-mse_graph_only,4))
assert math.isclose(mse_both_fg-mse_ligand_fg_only,-0.0102,abs_tol=1e-10)
assert math.isclose(mse_protein_fg_only-mse_graph_only,-0.0196,abs_tol=1e-10)
print('PASS: DAVIS single-split ablation is compatible with an inductive-bias/learning effect.')
print('LIMITATION: this does not prove new information, robust improvement, or KIBA CI causality.')


DAVIS Table 11: graph only 0.2018 ; protein FG only 0.1822 ; ligand FG only 0.1648 ; both FG 0.1546
Adding protein FG to ligand FG: MSE change -0.0102 ; percent change -6.189 %
Adding protein FG to graph only: MSE change -0.0196
PASS: DAVIS single-split ablation is compatible with an inductive-bias/learning effect.
LIMITATION: this does not prove new information, robust improvement, or KIBA CI causality.
